In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

%matplotlib widget
from PipelineOrtomosaico import PipelineOrtomosaico

# --- Configuración del vuelo ---
FECHA = "17ene"

# Opcional: pasar un polígono Shapely propio; None → selector interactivo / coords default
POLIGONO = None

# --- Ejecución del pipeline completo ---
resultado = PipelineOrtomosaico.procesar(
    fecha=FECHA,
    poligono=POLIGONO,
    mostrar_graficos=True,
    exportar_metricas=True,
    # Parámetros opcionales — se pueden omitir si se usan los defaults:
    indice_otsu="ndvi",
    tamano_minimo_pixeles=50,
    percentil_terreno=1.5,
    dilation_iterations=5,
    gaussian_sigma=15,
)

# --- Acceso rápido a los resultados ---
rgb         = resultado["rgb_final"]
ms          = resultado["ms_final"]
dsm         = resultado["dsm_final"]
indices     = resultado["indices_enmascarados"]
mascara     = resultado["mascara_binaria"]
chm         = resultado["chm"]
rugosidades = resultado["rugosidades"]

# Mapeo de bandas MS (Ajustar índices si tu sensor tiene otro orden)
bandas_ms = {
    "ms_blue":     ms[0],
    "ms_green":    ms[1],
    "ms_red":      ms[2],
    "ms_red_edge": ms[3],
    "ms_nir":      ms[4]
}

# Mapeo de bandas RGB
bandas_rgb = {
    "rgb_red":   rgb[0],
    "rgb_green": rgb[1],
    "rgb_blue":  rgb[2]
}

In [8]:
import pandas as pd
import numpy as np

# 1. Extraemos coordenadas de los píxeles de vegetación
coords_veg = np.where(mascara)
rows, cols = coords_veg

# 2. Inicializamos con la posición (Clave para detectar gradientes en el lote)
feature_dict = {
    'pos_x': cols,
    'pos_y': rows
}

# 3. Agregamos las Bandas MS crudas
for nombre, matriz in bandas_ms.items():
    feature_dict[nombre] = matriz[coords_veg]

# 4. Agregamos las Bandas RGB crudas
for nombre, matriz in bandas_rgb.items():
    feature_dict[nombre] = matriz[coords_veg]

# 5. Agregamos los Índices Espectrales
for nombre_idx, matriz_idx in indices.items():
    feature_dict[nombre_idx] = matriz_idx[coords_veg]

# 6. Agregamos Altura (CHM) y Rugosidades
if chm is not None:
    val_chm = chm[coords_veg]
    feature_dict['chm'] = val_chm
    
    # MÉTRICA DE INGENIERÍA: Vigor relativo a la altura (NDVI / CHM)
    # Útil para detectar plantas "estresadas pero altas" o "robustas pero bajas"
    if 'ndvi' in indices:
        feature_dict['ratio_ndvi_chm'] = indices['ndvi'][coords_veg] / (val_chm + 0.001)

if rugosidades:
    for nombre_rug, matriz_rug in rugosidades.items():
        feature_dict[f'rug_{nombre_rug.split()[0]}'] = matriz_rug[coords_veg]

# 7. Crear DataFrame y Limpiar
df_features = pd.DataFrame(feature_dict).dropna()

print(f"Feature Space listo: {df_features.shape[0]:,} píxeles x {df_features.shape[1]} columnas.")
display(df_features.head())

Feature Space listo: 90,282 píxeles x 23 columnas.


,pos_x,pos_y,ms_blue,ms_green,ms_red,ms_red_edge,ms_nir,rgb_red,rgb_green,rgb_blue,...,gndvi,vari,exg,gi,evi,chm,ratio_ndvi_chm,rug_25cm,rug_55cm,rug_105cm
0,708,164,0.013607,0.014112,0.050061,0.034885,1.0,0.392403,0.436567,0.326810,...,0.560190,0.087949,0.153922,0.310004,0.084174,0.033636,16.531407,0.104774,0.122862,0.104657
1,720,164,0.013886,0.015546,0.056342,0.041715,1.0,0.496640,0.502731,0.419416,...,0.567496,0.010502,0.089405,0.242924,0.097406,0.079877,7.474860,0.066897,0.076281,0.066638
2,707,165,0.012280,0.014324,0.070053,0.044288,1.0,0.328662,0.402941,0.275371,...,0.660476,0.162808,0.201848,0.367903,0.131361,0.181593,3.843009,0.127660,0.130249,0.105192
3,708,165,0.009838,0.012823,0.070562,0.047798,1.0,0.305693,0.458317,0.233251,...,0.692435,0.287557,0.377689,0.493276,0.138742,0.172990,4.340860,0.127177,0.128325,0.104695
4,709,165,0.012897,0.013159,0.052543,0.036145,1.0,0.333277,0.399181,0.273590,...,0.599432,0.143623,0.191495,0.362596,0.091475,0.106278,5.647267,0.123466,0.126266,0.103790
